In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time, random, re
from datetime import datetime

In [ ]:
BASE_URL   = "https://www.ambitionbox.com/reviews/ltimindtree-reviews"
MAX_PAGES  = None    
OUTPUT_CSV = "ltimindtree_reviews.csv"
DELAY_MIN  = 3.0
DELAY_MAX  = 5.0
HEADLESS   = True

opts = Options()
if HEADLESS:
    opts.add_argument("--headless=new")
opts.add_argument("--no-sandbox")
opts.add_argument("--disable-dev-shm-usage")
opts.add_argument("--disable-blink-features=AutomationControlled")
opts.add_experimental_option("excludeSwitches", ["enable-automation"])
opts.add_experimental_option("useAutomationExtension", False)
opts.add_argument("--window-size=1920,1080")
opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

def scroll_page(driver, pause=2.5):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
    time.sleep(pause / 2)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(pause)

def get_soup(driver):
    return BeautifulSoup(driver.page_source, "html.parser")

def extract_company_name(soup):
    h1 = soup.find("h1")
    if h1:
        text = re.sub(r"\s*(employee\s*)?reviews?.*$", "", h1.get_text(strip=True), flags=re.I).strip()
        if text:
            return text
    return "Unknown"

def find_review_cards(soup):
    likes_headings = soup.find_all("h3", class_=lambda c: c and "font-pn-600" in c,
                                   string=lambda t: t and t.strip() == "Likes")
    cards = []
    for h3 in likes_headings:
        node = h3.parent
        for _ in range(10):
            if node is None:
                break
            parent = node.parent
            if parent:
                siblings = [s for s in parent.children
                            if hasattr(s, "find") and
                            s.find("h3", string=lambda t: t and t.strip() == "Likes")]
                if len(siblings) >= 2:
                    cards.append(node)
                    break
            node = node.parent
    return cards

def get_text_after_heading(h3_node):
    sib = h3_node.find_next_sibling()
    if sib:
        return sib.get_text(" ", strip=True)
    parent = h3_node.parent
    if parent:
        full = parent.get_text(" ", strip=True)
        after = full[len(h3_node.get_text(strip=True)):].strip(" :-–")
        return after
    return ""

def extract_rating(card):
    for el in card.find_all(["span", "div", "p"]):
        txt = el.get_text(strip=True)
        if re.fullmatch(r"[1-5](\.\d)?", txt):
            return txt
    return ""

def extract_date(card):
    full_text = card.get_text(" ", strip=True)
    patterns = [
        (r"\b(\d{1,2}\s+\w+\s+\d{4})\b", ["%d %b %Y", "%d %B %Y"]),
        (r"\b(\w+\s+\d{4})\b",            ["%b %Y", "%B %Y"]),
        (r"\b(\d{4}-\d{2}-\d{2})\b",      ["%Y-%m-%d"]),
    ]
    for pattern, fmts in patterns:
        m = re.search(pattern, full_text)
        if m:
            raw = m.group(1)
            for fmt in fmts:
                try:
                    return datetime.strptime(raw, fmt).strftime("%d-%b-%y")
                except ValueError:
                    pass
    time_el = card.find("time")
    if time_el:
        raw = time_el.get("datetime") or time_el.get_text(strip=True)
        for fmt in ["%Y-%m-%d", "%d %b %Y", "%b %Y"]:
            try:
                return datetime.strptime(raw.strip(), fmt).strftime("%d-%b-%y")
            except ValueError:
                pass
    return ""

def parse_card(card, company, page, idx, base_url):
    likes_h3    = card.find("h3", string=lambda t: t and t.strip() == "Likes")
    dislikes_h3 = card.find("h3", string=lambda t: t and t.strip() == "Dislikes")
    like_text    = get_text_after_heading(likes_h3)    if likes_h3    else ""
    dislike_text = get_text_after_heading(dislikes_h3) if dislikes_h3 else ""
    return {
        "company":              company,
        "page":                 page,
        "review_index_on_page": idx,
        "like_text":            like_text,
        "dislike_text":         dislike_text,
        "overall_rating":       extract_rating(card),
        "review_date":          extract_date(card),
        "review_url":           base_url,
    }

def build_url(base, page_num):
    base = re.sub(r"[?&]page=\d+", "", base).rstrip("/")
    return base if page_num == 1 else f"{base}?page={page_num}"

def get_next_page_url(soup, current_page_num):
    """Find the link for current_page_num + 1 in the pagination links."""
    next_num = current_page_num + 1
    # Look for <a href="...?page=NEXT">
    link = soup.find("a", href=re.compile(rf"[?&]page={next_num}$"))
    if link:
        href = link["href"]
        return href if href.startswith("http") else f"https://www.ambitionbox.com{href}"
    return None

all_rows = []
company  = None

try:
    for page_num in range(1, (MAX_PAGES or 9999) + 1):
        url = build_url(BASE_URL, page_num)
        print(f"\nPage {page_num}: {url}")
        driver.get(url)

        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.XPATH, "//h3[normalize-space()='Likes']"))
            )
        except Exception:
            print("  Timeout – no 'Likes' headings found.")

        scroll_page(driver, pause=2.5)
        soup = get_soup(driver)

        if company is None:
            company = extract_company_name(soup)
            print(f"  Company: {company}")

        cards = find_review_cards(soup)
        print(f"  Cards found: {len(cards)}")

        if not cards:
            print("  No cards – stopping.")
            break

        for i, card in enumerate(cards, 1):
            all_rows.append(parse_card(card, company, page_num, i, BASE_URL))

        df = pd.DataFrame(all_rows, columns=[
            "company","page","review_index_on_page",
            "like_text","dislike_text","overall_rating",
            "review_date","review_url",
        ])
        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        print(f"  Running total: {len(all_rows)} rows  →  {OUTPUT_CSV}")

        next_url = get_next_page_url(soup, page_num)
        if not next_url:
            print("  No next page. Done.")
            break

        print(f"  Next → {next_url}")
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

finally:
    driver.quit()

print(f"\n✅ Scrape complete! Total reviews: {len(all_rows)}")

df = pd.DataFrame(all_rows, columns=[
    "company","page","review_index_on_page",
    "like_text","dislike_text","overall_rating",
    "review_date","review_url",
])
df


Page 1: https://www.ambitionbox.com/reviews/ltimindtree-reviews
